# EX: Building a ReAct Loop Agent

In this exercise, we will build a basic ReAct loop agent from scratch using Python. We will intentionally inject a tool error to observe how the agent utilizes its short-term memory to recover and accomplish the mission.

Here are the specific steps we will implement:

* **Define the Simulated Tools:** Create mock Python functions representing external capabilities (a weather API and an airspace radar) that the agent can execute.*

* **Simulate the LLM Output:** Instead of using a live API key, we will hardcode a sequence of JSON responses to represent the agent's internal reasoning (Thought) and chosen actions (Action).

* **Build the ReAct Orchestrator:** Construct the execution while loop that acts as the agent's brain, parsing thoughts, calling the appropriate tools, and capturing the external results (Observation).

* **Execute and Observe Error Recovery:** We will run the agent through a scenario where its primary tool fails on "Grid-B," forcing it to reflect, adjust its plan, and successfully pivot to a secondary drop zone.

In [ ]:
# Only run this cell after downloading and selecting your kernel
!python.exe -m pip install --upgrade pip
!pip install json

In [1]:

import json

# 1. Define the Simulated Tools (The Agent's capabilities)
def check_weather(location):
    """Simulates checking weather. Fails intentionally on 'Grid-B'."""
    print(f"   [SYSTEM] Executing tool 'check_weather' for {location}...")
    if location == "Grid-B":
        return "ERROR: Weather API Offline for Grid-B. Try alternative location."
    return "Weather is CLEAR. Wind 5 knots."

def check_airspace(location):
    """Simulates checking if airspace is clear."""
    print(f"   [SYSTEM] Executing tool 'check_airspace' for {location}...")
    return "Airspace is RESTRICTED. Hostile anti-air activity detected."

# A dictionary mapping tool names to their actual Python functions
AVAILABLE_TOOLS = {
    "check_weather": check_weather,
    "check_airspace": check_airspace
}

# 2. Simulate the LLM's Output (For demonstration without an API key)
# In a real system, an LLM generates these responses dynamically.
# Here, we hardcode the LLM's sequential responses to demonstrate the ReAct logic flow.
llm_responses = [
    # Attempt 1: The LLM decides to check the weather at the primary target.
    '{"Thought": "I need to plan an airdrop at Grid-B. I must first check the weather.", "Action": "check_weather", "Action_Input": "Grid-B"}',
    
    # Attempt 2: The LLM reflects on the error from Attempt 1 and adjusts its plan.
    '{"Thought": "The weather API failed for Grid-B. I cannot risk a drop there without weather data. I will switch to the secondary drop zone, Grid-A, and check its weather.", "Action": "check_weather", "Action_Input": "Grid-A"}',
    
    # Attempt 3: Weather at Grid-A is clear. Now the LLM checks airspace.
    '{"Thought": "Weather at Grid-A is clear. Now I must ensure the airspace is safe for the transport aircraft.", "Action": "check_airspace", "Action_Input": "Grid-A"}',
    
    # Attempt 4: The LLM receives the hostile airspace warning and makes a final decision.
    '{"Thought": "Airspace at Grid-A is restricted due to hostile activity. The mission is unsafe.", "Action": "FINAL_ANSWER", "Action_Input": "Airdrop mission ABORTED. Grid-B weather unknown; Grid-A airspace restricted."}'
]

# 3. The ReAct Execution Loop (The Orchestrator)
def run_react_agent(mission_prompt):
    print(f"--- AGENT ACTIVATED ---\nMission: {mission_prompt}\n")
    
    # The agent's short-term memory (context window)
    memory = [] 
    
    step_count = 0
    max_steps = 5 # Safeguard against infinite loops
    
    while step_count < max_steps:
        print(f"--- STEP {step_count + 1} ---")
        
        # In a real app: response = call_llm(prompt + memory)
        raw_response = llm_responses[step_count] 
        parsed_response = json.loads(raw_response)
        
        thought = parsed_response["Thought"]
        action_name = parsed_response["Action"]
        action_input = parsed_response["Action_Input"]
        
        print(f"🤔 THOUGHT: {thought}")
        
        # Check if the agent has reached a conclusion
        if action_name == "FINAL_ANSWER":
            print(f"✅ FINAL PLAN: {action_input}\n")
            break
            
        print(f"🛠️ ACTION:  Call '{action_name}' with parameter '{action_input}'")
        
        # Execute the tool and capture the observation
        tool_function = AVAILABLE_TOOLS.get(action_name)
        if tool_function:
            observation = tool_function(action_input)
        else:
            observation = f"ERROR: Tool '{action_name}' not found."
            
        print(f"👁️ OBSERVATION: {observation}\n")
        
        # Store the cycle in memory so the agent can read it on the next loop
        memory.append({
            "Thought": thought, 
            "Action": action_name, 
            "Observation": observation
        })
        
        step_count += 1

# Execute the simulation
run_react_agent("Plan a supply airdrop at primary target Grid-B. If unsafe, use secondary target Grid-A.")



--- AGENT ACTIVATED ---
Mission: Plan a supply airdrop at primary target Grid-B. If unsafe, use secondary target Grid-A.

--- STEP 1 ---
🤔 THOUGHT: I need to plan an airdrop at Grid-B. I must first check the weather.
🛠️ ACTION:  Call 'check_weather' with parameter 'Grid-B'
   [SYSTEM] Executing tool 'check_weather' for Grid-B...
👁️ OBSERVATION: ERROR: Weather API Offline for Grid-B. Try alternative location.

--- STEP 2 ---
🤔 THOUGHT: The weather API failed for Grid-B. I cannot risk a drop there without weather data. I will switch to the secondary drop zone, Grid-A, and check its weather.
🛠️ ACTION:  Call 'check_weather' with parameter 'Grid-A'
   [SYSTEM] Executing tool 'check_weather' for Grid-A...
👁️ OBSERVATION: Weather is CLEAR. Wind 5 knots.

--- STEP 3 ---
🤔 THOUGHT: Weather at Grid-A is clear. Now I must ensure the airspace is safe for the transport aircraft.
🛠️ ACTION:  Call 'check_airspace' with parameter 'Grid-A'
   [SYSTEM] Executing tool 'check_airspace' for Grid-A...
👁️ O

## Interpreting the Results

When you run this script, you witness the ReAct loop in action. The agent attempts to use the check_weather tool on Grid-B, which intentionally fails. A fragile script would crash here. However, the agent's Reflection capability allows it to read the error observation, generate a new Thought ("I will switch to the secondary drop zone"), and continue problem-solving until it reaches a logical, safe conclusion to abort the mission.